# Medical Information Extraction Lab - Offline Clinical NLP

Notebook end-to-end cho Clinical NER, assertion/context, ICD-10/RxNorm linking, relation baseline, evaluator và submission.

## Chạy trên Google Colab

1. Chuẩn bị dữ liệu một lần tại `MyDrive/AI-Race-Viettel/data/` theo cấu trúc ghi trong `COLAB_RUNBOOK.md`.
2. Upload riêng file `.ipynb`; cell đầu tiên tự mount Drive, clone nhánh `Pipeline_colab` và cài dependencies.
3. Notebook tự tìm input thật, annotation, knowledge sources và lưu checkpoint/output về Drive.
4. Giữ `FAST_DEV_RUN = False` để train đầy đủ; đổi thành `True` cho smoke-test ngắn.
5. Bootstrap có thể tạo smoke input tạm để hoàn tất khởi tạo, nhưng production gate ngay sau đó vẫn bắt buộc input thật và sẽ dừng với hướng dẫn rõ; smoke input không bao giờ được dùng để tạo ZIP production.

Notebook không gọi external API và không fit trên private/test input. Hướng dẫn vận hành được giữ trong `README.md` và `COLAB_RUNBOOK.md`.

## 0. Project overview

**Mục tiêu:** Chuyển văn bản lâm sàng phi cấu trúc thành entity JSON có span, type, assertions và candidate chuẩn hóa.

**Pipeline:** raw text -> validation -> sectioning -> detection -> span refinement -> context -> type-routed linking -> relation diagnostics -> schema conversion -> submission validation -> `output.zip`.

**Dữ liệu train:** nếu Drive chưa có annotation, supervised training được bỏ qua minh bạch và inference dùng artifacts/rule-dictionary baseline. Official entity/assertion schema được đối chiếu từ validator trong chính repository.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

# One-click Colab bootstrap. The defaults clone the committed Colab branch.
IS_COLAB = "COLAB_RELEASE_TAG" in os.environ
MOUNT_GOOGLE_DRIVE = True
INSTALL_REQUIREMENTS = True
COLAB_REQUIREMENTS_FILE = "requirements-colab.txt"
FAST_DEV_RUN = False
AUTO_SMOKE_INPUT = True
PROJECT_ROOT_OVERRIDE = ""
GITHUB_REPO_URL = "https://github.com/takumi612/AI-Race-Viettel.git"
GITHUB_BRANCH = "Pipeline_colab"
INPUT_ZIP_OVERRIDE = ""
TRAIN_DIR_OVERRIDE = ""
ICD10_PATH_OVERRIDE = ""
RXNORM_ZIP_OVERRIDE = ""
TRAINING_OUTPUT_DIR_OVERRIDE = ""
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/clinical-nlp-end-to-end-lab")
DRIVE_TRAINING_OUTPUT_DIR = Path("/content/drive/MyDrive/clinical-nlp-training-artifacts")
COLAB_REPO_DIR = Path("/content/AI-Race-Viettel")

if IS_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

if IS_COLAB and GITHUB_REPO_URL.strip() and not COLAB_REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(COLAB_REPO_DIR)],
        check=True,
    )

project_candidates = []
if PROJECT_ROOT_OVERRIDE.strip():
    project_candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())
project_candidates.extend([
    COLAB_REPO_DIR / "Code_E_Platform",
    COLAB_REPO_DIR,
    DRIVE_PROJECT_DIR / "Code_E_Platform",
    DRIVE_PROJECT_DIR,
    Path("/content/clinical-nlp-end-to-end-lab/Code_E_Platform"),
    Path("/content/clinical-nlp-end-to-end-lab"),
    Path("/content/AI_race/Code_E_Platform"),
    Path("/content/AI_race"),
    Path.cwd() / "Code_E_Platform",
    Path.cwd(),
])
PROJECT_ROOT = next(
    (candidate.resolve() for candidate in project_candidates if (candidate / "clinical_nlp_lab").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Automatic clone failed; set PROJECT_ROOT_OVERRIDE."
    )

if TRAINING_OUTPUT_DIR_OVERRIDE.strip():
    TRAINING_OUTPUT_ROOT = Path(TRAINING_OUTPUT_DIR_OVERRIDE).expanduser().resolve()
elif IS_COLAB and MOUNT_GOOGLE_DRIVE:
    TRAINING_OUTPUT_ROOT = DRIVE_TRAINING_OUTPUT_DIR
else:
    TRAINING_OUTPUT_ROOT = PROJECT_ROOT / "artifacts"
TRAINING_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if IS_COLAB and INSTALL_REQUIREMENTS:
    requirements_path = PROJECT_ROOT / COLAB_REQUIREMENTS_FILE
    if not requirements_path.exists():
        requirements_path = PROJECT_ROOT / "requirements.txt"
    if requirements_path.exists():
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
            check=True,
        )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from clinical_nlp_lab.config import load_config, set_reproducible_seed
from clinical_nlp_lab.data import (
    describe_documents,
    document_train_validation_split,
    load_annotated_documents,
    load_input_documents,
    validate_documents,
)
from clinical_nlp_lab.kb import load_candidate_dictionary
from clinical_nlp_lab.ner import DictionaryRuleEntityDetector, refine_boundaries
from clinical_nlp_lab.assertions import HybridAssertionPredictor
from clinical_nlp_lab.linking import EntityLinker, LexicalCandidateIndex
from clinical_nlp_lab.relations import RuleRelationExtractor
from clinical_nlp_lab.pipeline import reload_equivalence_check, run_inference
from clinical_nlp_lab.schema import write_json

def _has_text_files(path: Path) -> bool:
    return path.is_dir() and any(path.glob("*.txt"))

def _has_annotation_files(path: Path) -> bool:
    return path.is_dir() and (any(path.glob("*.json")) or any(path.glob("*.txt")))

CONFIG = load_config(PROJECT_ROOT / "artifacts/config.json")
CONFIG["fast_dev_run"] = FAST_DEV_RUN
for config_key, override_value in {
    "input_zip": INPUT_ZIP_OVERRIDE,
    "train_dir": TRAIN_DIR_OVERRIDE,
    "icd10_path": ICD10_PATH_OVERRIDE,
    "rxnorm_zip_path": RXNORM_ZIP_OVERRIDE,
}.items():
    if override_value.strip():
        CONFIG[config_key] = override_value

input_candidates = []
if INPUT_ZIP_OVERRIDE.strip():
    input_candidates.append(Path(INPUT_ZIP_OVERRIDE).expanduser())
input_candidates.extend([
    PROJECT_ROOT / "input.zip",
    PROJECT_ROOT / "data/input",
    PROJECT_ROOT.parent / "input.zip",
    PROJECT_ROOT.parent / "data/input",
    DRIVE_PROJECT_DIR / "input.zip",
    DRIVE_PROJECT_DIR / "data/input",
])
INPUT_SOURCE = next((candidate for candidate in input_candidates if candidate.is_file() or _has_text_files(candidate)), None)
SMOKE_INPUT_USED = False
if INPUT_SOURCE is None and AUTO_SMOKE_INPUT:
    smoke_root = Path(tempfile.gettempdir()) / "clinical_nlp_smoke_input"
    smoke_root.mkdir(parents=True, exist_ok=True)
    (smoke_root / "1.txt").write_text(
        "HISTORY\nPatient reports fever and cough.\nMEDICATIONS\nAspirin 81 mg po daily.",
        encoding="utf-8",
    )
    INPUT_SOURCE = smoke_root
    SMOKE_INPUT_USED = True
if INPUT_SOURCE is None:
    raise FileNotFoundError("No input source found. Set INPUT_ZIP_OVERRIDE or enable AUTO_SMOKE_INPUT.")
CONFIG["input_zip"] = str(INPUT_SOURCE)

train_candidates = []
if TRAIN_DIR_OVERRIDE.strip():
    train_candidates.append(Path(TRAIN_DIR_OVERRIDE).expanduser())
train_candidates.extend([PROJECT_ROOT / "train", DRIVE_PROJECT_DIR / "train"])
TRAIN_SOURCE = next((candidate for candidate in train_candidates if _has_annotation_files(candidate)), None)
if TRAIN_SOURCE is None:
    TRAIN_SOURCE = PROJECT_ROOT / "train"
CONFIG["train_dir"] = str(TRAIN_SOURCE)

SEED_STATUS = set_reproducible_seed(int(CONFIG["seed"]))
print({"is_colab": IS_COLAB, "project_root": str(PROJECT_ROOT), "input_source": str(INPUT_SOURCE), "smoke_input_used": SMOKE_INPUT_USED, "train_source": str(TRAIN_SOURCE), "training_output_root": str(TRAINING_OUTPUT_ROOT), "seed_status": SEED_STATUS, "fast_dev_run": CONFIG["fast_dev_run"], "cuda": os.environ.get("CUDA_VISIBLE_DEVICES", "auto")})

## 0.1 Production data and output resolver

Cell này là production gate cho chế độ `Run all`. Nó ưu tiên dữ liệu thật trên Google Drive, hỗ trợ cả `input.zip` và thư mục `input/*.txt`, nhận annotation dạng `train/*.txt + *.json` hoặc `synthetic_train_v1/input + gt`, và lưu `output.zip` về Drive. Nếu không tìm thấy input thật, notebook dừng ngay với cấu trúc thư mục cần tạo.

In [ ]:
# Production defaults: upload data once to Drive, then Run all needs no edits.
REQUIRE_REAL_INPUT = True
DATA_ROOT_OVERRIDE = ""
OUTPUT_ARCHIVE_OVERRIDE = ""
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/AI-Race-Viettel/data")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/AI-Race-Viettel/output")

def _unique_paths(paths):
    seen = set()
    result = []
    for path in paths:
        normalized = str(path)
        if normalized not in seen:
            seen.add(normalized)
            result.append(path)
    return result

data_roots = []
if DATA_ROOT_OVERRIDE.strip():
    data_roots.append(Path(DATA_ROOT_OVERRIDE).expanduser())
data_roots.extend([
    DRIVE_DATA_DIR,
    Path("/content/drive/MyDrive/clinical-nlp-data"),
    DRIVE_PROJECT_DIR / "data",
    PROJECT_ROOT / "data",
    PROJECT_ROOT.parent / "data",
    PROJECT_ROOT.parent,
    PROJECT_ROOT,
])
data_roots = _unique_paths(data_roots)

input_candidates = []
if INPUT_ZIP_OVERRIDE.strip():
    input_candidates.append(Path(INPUT_ZIP_OVERRIDE).expanduser())
for root in data_roots:
    input_candidates.extend([root / "input.zip", root / "input", root / "data/input"])
REAL_INPUT_SOURCE = next(
    (path for path in _unique_paths(input_candidates) if path.is_file() or _has_text_files(path)),
    None,
)
if REAL_INPUT_SOURCE is None and REQUIRE_REAL_INPUT:
    raise FileNotFoundError(
        "Real input not found. Create MyDrive/AI-Race-Viettel/data/input/ with .txt files "
        "or upload MyDrive/AI-Race-Viettel/data/input.zip. See COLAB_RUNBOOK.md."
    )
if REAL_INPUT_SOURCE is not None:
    INPUT_SOURCE = REAL_INPUT_SOURCE
    CONFIG["input_zip"] = str(INPUT_SOURCE)
    SMOKE_INPUT_USED = False

def _has_training_layout(path: Path) -> bool:
    if not path.is_dir():
        return False
    text_stems = {item.stem for item in path.glob("*.txt")}
    json_stems = {item.stem for item in path.glob("*.json")}
    direct_pairs = bool(text_stems & json_stems)
    split_pairs = _has_text_files(path / "input") and (path / "gt").is_dir() and any((path / "gt").glob("*.json"))
    return direct_pairs or split_pairs

train_candidates = []
if TRAIN_DIR_OVERRIDE.strip():
    train_candidates.append(Path(TRAIN_DIR_OVERRIDE).expanduser())
for root in data_roots:
    train_candidates.extend([root / "train", root / "synthetic_train_v1", root])
REAL_TRAIN_SOURCE = next(
    (path for path in _unique_paths(train_candidates) if _has_training_layout(path)),
    None,
)
if REAL_TRAIN_SOURCE is not None:
    TRAIN_SOURCE = REAL_TRAIN_SOURCE
    CONFIG["train_dir"] = str(TRAIN_SOURCE)

source_overrides = {
    "icd10_path": (ICD10_PATH_OVERRIDE, "ICD10.xlsx"),
    "rxnorm_zip_path": (RXNORM_ZIP_OVERRIDE, "RxNorm_full_07062026.zip"),
}
for config_key, (explicit_path, filename) in source_overrides.items():
    candidates = [Path(explicit_path).expanduser()] if explicit_path.strip() else []
    candidates.extend(root / filename for root in data_roots)
    discovered = next((path for path in _unique_paths(candidates) if path.is_file()), None)
    if discovered is not None:
        CONFIG[config_key] = str(discovered)

if OUTPUT_ARCHIVE_OVERRIDE.strip():
    OUTPUT_ARCHIVE_PATH = Path(OUTPUT_ARCHIVE_OVERRIDE).expanduser()
elif IS_COLAB and MOUNT_GOOGLE_DRIVE:
    OUTPUT_ARCHIVE_PATH = DRIVE_OUTPUT_DIR / "output.zip"
else:
    OUTPUT_ARCHIVE_PATH = PROJECT_ROOT / "output.zip"
OUTPUT_ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)

print({
    "real_input": str(INPUT_SOURCE),
    "training_data": str(REAL_TRAIN_SOURCE) if REAL_TRAIN_SOURCE else "not_found_training_will_skip",
    "output_zip": str(OUTPUT_ARCHIVE_PATH),
    "icd10_source": CONFIG["icd10_path"],
    "rxnorm_source": CONFIG["rxnorm_zip_path"],
})

## 1. Environment setup

**Mục tiêu:** Kiểm tra runtime và optional supervised dependencies.  
**Công nghệ:** Python stdlib, package-local modules; optional torch/transformers.  
**Input:** Project root and requirements.txt.  
**Output:** CONFIG and dependency status.  
**Artifact:** No external API or secret.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
from clinical_nlp_lab.training import transformer_training_availability
TRAINING_AVAILABILITY = transformer_training_availability()
print({"python": sys.version, "training": TRAINING_AVAILABILITY.reason})

## 2. Configuration

**Mục tiêu:** Giữ mọi đường dẫn, model, threshold và resource filter trong một object.  
**Công nghệ:** CONFIG dict and JSON artifact.  
**Input:** artifacts/config.json.  
**Output:** CONFIG.  
**Artifact:** artifacts/config.json.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
from clinical_nlp_lab.schema import ALLOWED_ASSERTIONS, OFFICIAL_SCHEMA_KEYS

assert CONFIG["max_length"] == 512
assert CONFIG["stride"] == 128
assert OFFICIAL_SCHEMA_KEYS["CHẨN_ĐOÁN"] == {"text", "type", "position", "assertions", "candidates"}
assert OFFICIAL_SCHEMA_KEYS["TRIỆU_CHỨNG"] == {"text", "type", "position", "assertions"}
assert ALLOWED_ASSERTIONS == {"isNegated", "isHistorical", "isFamily"}
print(json.dumps({"config_keys": len(CONFIG), "thresholds": CONFIG["thresholds"], "official_schema": {key: sorted(value) for key, value in OFFICIAL_SCHEMA_KEYS.items()}}, ensure_ascii=True))

## 3. Data discovery

**Mục tiêu:** Đọc đúng ZIP/text và kiểm tra annotated split.  
**Công nghệ:** pathlib, zipfile, UTF-8 loader.  
**Input:** input.zip and train/ if present.  
**Output:** DOCUMENTS and ANNOTATED_DOCUMENTS.  
**Artifact:** Data fingerprint in validation report.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
DOCUMENTS = load_input_documents(PROJECT_ROOT / CONFIG["input_zip"])
ANNOTATED_DOCUMENTS = load_annotated_documents(PROJECT_ROOT / CONFIG["train_dir"])
print({"input_documents": len(DOCUMENTS), "annotated_documents": len(ANNOTATED_DOCUMENTS)})

## 4. Data validation

**Mục tiêu:** Kiểm tra schema, offset, duplicate và overlap.  
**Công nghệ:** Custom validator with raw-text slice invariant.  
**Input:** DOCUMENTS and ANNOTATED_DOCUMENTS.  
**Output:** INPUT_VALIDATION and ANNOTATED_VALIDATION.  
**Artifact:** In-memory validation report.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
INPUT_VALIDATION = validate_documents(DOCUMENTS)
ANNOTATED_VALIDATION = validate_documents(ANNOTATED_DOCUMENTS)
assert INPUT_VALIDATION["is_valid"]
print({"input_validation": INPUT_VALIDATION, "annotation_count": len(ANNOTATED_DOCUMENTS)})

## 5. Exploratory data analysis

**Mục tiêu:** Tóm tắt độ dài và cấu trúc, không học tham số từ private input.  
**Công nghệ:** describe_documents and section counters.  
**Input:** DOCUMENTS.  
**Output:** EDA_SUMMARY.  
**Artifact:** In-memory EDA summary.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
EDA_SUMMARY = describe_documents(DOCUMENTS)
print(EDA_SUMMARY)

## 6. Train/validation split

**Mục tiêu:** Split theo document trước mọi chunking/fitting.  
**Công nghệ:** Deterministic random seed.  
**Input:** ANNOTATED_DOCUMENTS only.  
**Output:** TRAIN_DOCUMENTS and VALIDATION_DOCUMENTS.  
**Artifact:** No split artifact if annotations are absent.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
TRAIN_DOCUMENTS, VALIDATION_DOCUMENTS = document_train_validation_split(
    ANNOTATED_DOCUMENTS, CONFIG["validation_fraction"], int(CONFIG["seed"])
)
assert not ({item.document_id for item in TRAIN_DOCUMENTS} & {item.document_id for item in VALIDATION_DOCUMENTS})
print({"train": len(TRAIN_DOCUMENTS), "validation": len(VALIDATION_DOCUMENTS), "leakage": False})

## 7. Section detection

**Mục tiêu:** Giữ section name/start/end/text trên raw text.  
**Công nghệ:** Regex heading dictionary and newline-aware segmentation.  
**Input:** DOCUMENTS[0].raw_text.  
**Output:** SECTION_SAMPLE.  
**Artifact:** Section spans in diagnostics.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
from clinical_nlp_lab.text import detect_sections
SECTION_SAMPLE = detect_sections(DOCUMENTS[0].raw_text)
for section in SECTION_SAMPLE:
    section.validate(DOCUMENTS[0].raw_text)
print({"sections": len(SECTION_SAMPLE), "names": [s.section_name for s in SECTION_SAMPLE]})

## 8. ICD-10 preprocessing

**Mục tiêu:** Load/cache bilingual ICD-10 candidates and canonicalize markers.  
**Công nghệ:** ICD-10 JSONL.GZ cache; build script if cache is missing.  
**Input:** ICD10.xlsx.  
**Output:** ICD10_RECORDS.  
**Artifact:** artifacts/icd10/.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
icd10_cache = PROJECT_ROOT / "artifacts/icd10/icd10_dictionary.jsonl.gz"
if not icd10_cache.exists():
    subprocess.run([sys.executable, str(PROJECT_ROOT / "tools/build_knowledge_bases.py"), "--root", str(PROJECT_ROOT), "--artifact-dir", str(PROJECT_ROOT / "artifacts")], check=True)
ICD10_RECORDS = load_candidate_dictionary(icd10_cache)
print({"icd10_candidates": len(ICD10_RECORDS)})

## 9. RxNorm preprocessing

**Mục tiêu:** Load filtered RxNorm and optional relation cache without full extraction.  
**Công nghệ:** Streaming RRF parser and compressed cache.  
**Input:** RxNorm_full_07062026.zip.  
**Output:** RXNORM_RECORDS and relation cache.  
**Artifact:** artifacts/rxnorm/.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
RXNORM_RECORDS = load_candidate_dictionary(PROJECT_ROOT / "artifacts/rxnorm/rxnorm_dictionary.jsonl.gz")
print({"rxnorm_candidates": len(RXNORM_RECORDS), "relation_cache": (PROJECT_ROOT / "artifacts/rxnorm/rxnorm_relations.jsonl.gz").exists()})

## 10. BIO or span dataset construction

**Mục tiêu:** Prepare character annotations for BIO labels and chunk windows.  
**Công nghệ:** BIO conversion, offsets and sliding windows.  
**Input:** TRAIN_DOCUMENTS.  
**Output:** BIO_LABEL_TO_ID and feature plan.  
**Artifact:** Conditional NER model artifact.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
from clinical_nlp_lab.training import build_bio_label_map, chunk_token_indices
ENTITY_TYPES = {entity.type for document in TRAIN_DOCUMENTS for entity in document.entities}
BIO_LABEL_TO_ID, BIO_ID_TO_LABEL = build_bio_label_map(ENTITY_TYPES)
print({"bio_labels": BIO_LABEL_TO_ID, "window_example": chunk_token_indices(1200, CONFIG["max_length"], CONFIG["stride"])})

## 11. NER training

**Mục tiêu:** Train XLM-R only when annotations and optional packages exist.  
**Công nghệ:** AutoTokenizer, AutoModelForTokenClassification, Trainer (guarded).  
**Input:** TRAIN_DOCUMENTS and VALIDATION_DOCUMENTS.  
**Output:** NER_TRAINING_RESULT.  
**Artifact:** artifacts/ner_model/ when trained.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
from clinical_nlp_lab.training import train_transformer_ner
FIT_TRAIN_DOCUMENTS = TRAIN_DOCUMENTS[:2] if CONFIG["fast_dev_run"] else TRAIN_DOCUMENTS
FIT_VALIDATION_DOCUMENTS = VALIDATION_DOCUMENTS[:1] if CONFIG["fast_dev_run"] else VALIDATION_DOCUMENTS
NER_MODEL_DIR = TRAINING_OUTPUT_ROOT / "ner_model"
NER_TRAINING_RESULT = train_transformer_ner(
    FIT_TRAIN_DOCUMENTS, FIT_VALIDATION_DOCUMENTS, NER_MODEL_DIR,
    model_name=CONFIG["ner_model_name"], max_length=CONFIG["max_length"], stride=CONFIG["stride"],
    learning_rate=CONFIG["learning_rate"], epochs=1 if CONFIG["fast_dev_run"] else CONFIG["ner_epochs"],
    batch_size=CONFIG["batch_size"], seed=CONFIG["seed"]
)
print({"fast_dev_run": CONFIG["fast_dev_run"], "fit_train_documents": len(FIT_TRAIN_DOCUMENTS), "fit_validation_documents": len(FIT_VALIDATION_DOCUMENTS), "model_dir": str(NER_MODEL_DIR), "result": NER_TRAINING_RESULT})

## 12. NER evaluation

**Mục tiêu:** Exact-span, relaxed-span and type metrics when gold exists.  
**Công nghệ:** Custom evaluator.  
**Input:** Gold/predicted annotations.  
**Output:** NER_EVALUATION.  
**Artifact:** Validation metrics only when annotations exist.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
NER_EVALUATION = {"status": "not_scored", "reason": "No annotated validation data"} if not VALIDATION_DOCUMENTS else {"status": "run_after_training"}
print(NER_EVALUATION)

## 13. Character span reconstruction

**Mục tiêu:** Reconstruct entity spans directly from raw offsets.  
**Công nghệ:** BIO-to-span converter and offset assertions.  
**Input:** Token offsets and BIO predictions.  
**Output:** Reconstructed spans.  
**Artifact:** No normalized raw text is used for final output.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
from clinical_nlp_lab.training import bio_predictions_to_spans
print({"offset_contract": "raw_text[start:end] == entity.text", "status": "implemented"})

## 14. Boundary refinement

**Mục tiêu:** Trim invalid whitespace and resolve overlapping spans deterministically.  
**Công nghệ:** Confidence ranking and type-aware overlap resolution.  
**Input:** Detector spans.  
**Output:** Refined spans.  
**Artifact:** Diagnostics offset checks.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
detector = DictionaryRuleEntityDetector(ICD10_RECORDS, RXNORM_RECORDS)
REFINED_SAMPLE = refine_boundaries(detector.detect(DOCUMENTS[0].raw_text), DOCUMENTS[0].raw_text)
for entity in REFINED_SAMPLE:
    entity.validate_offset(DOCUMENTS[0].raw_text)
print({"sample_spans": len(REFINED_SAMPLE), "offset_errors": 0})

## 15. Assertion dataset

**Mục tiêu:** Create entity-marked context examples for four assertion axes.  
**Công nghệ:** build_assertion_examples and section feature.  
**Input:** TRAIN_DOCUMENTS.  
**Output:** ASSERTION_EXAMPLES.  
**Artifact:** No supervised artifact when labels absent.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
from clinical_nlp_lab.training import build_assertion_examples
ASSERTION_EXAMPLES = build_assertion_examples(TRAIN_DOCUMENTS)
print({"assertion_examples": len(ASSERTION_EXAMPLES)})

## 16. Assertion training

**Mục tiêu:** Provide shared encoder/multi-head training factory without inventing labels.  
**Công nghệ:** Optional torch/transformers multi-task model.  
**Input:** ASSERTION_EXAMPLES when present.  
**Output:** Assertion model status.  
**Artifact:** artifacts/assertion_model/ only when trained.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
ASSERTION_TRAINING = {"trained": False, "reason": "No assertion labels in workspace"}
print(ASSERTION_TRAINING)

## 17. Assertion evaluation

**Mục tiêu:** Evaluate Jaccard and axis macro-F1 only with gold assertions.  
**Công nghệ:** Custom Jaccard and axis metrics.  
**Input:** Gold/predicted assertions.  
**Output:** ASSERTION_EVALUATION.  
**Artifact:** No score when annotations absent.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
ASSERTION_EVALUATION = {"status": "not_scored", "reason": "No assertion ground truth"}
print(ASSERTION_EVALUATION)

## 18. Candidate knowledge bases

**Mục tiêu:** Instantiate separate ICD-10 and RxNorm indexes.  
**Công nghệ:** LexicalCandidateIndex and type routing.  
**Input:** ICD10_RECORDS and RXNORM_RECORDS.  
**Output:** ICD_INDEX and RXNORM_INDEX.  
**Artifact:** Candidate index objects rebuilt from artifacts.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
ICD_INDEX = LexicalCandidateIndex(ICD10_RECORDS, "ICD-10")
RXNORM_INDEX = LexicalCandidateIndex(RXNORM_RECORDS, "RxNorm")
print({"icd10": len(ICD_INDEX.records), "rxnorm": len(RXNORM_INDEX.records)})

## 19. ICD-10 retrieval

**Mục tiêu:** Retrieve disease candidates using exact/fuzzy/character methods.  
**Công nghệ:** LexicalCandidateIndex with character n-gram similarity.  
**Input:** Disease mention.  
**Output:** Top-k ICD-10 candidates.  
**Artifact:** ICD-10 dictionary/cache.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
sample_icd = ICD10_RECORDS[0]
print(ICD_INDEX.retrieve((sample_icd.get("detection_aliases") or [sample_icd["candidate_id"]])[0], top_k=3))

## 20. RxNorm retrieval

**Mục tiêu:** Retrieve drug candidates and parse medication attributes.  
**Công nghệ:** RxNorm lexical index and medication parser.  
**Input:** Drug mention.  
**Output:** Top-k RXCUI candidates and attributes.  
**Artifact:** RxNorm dictionary/cache.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
from clinical_nlp_lab.linking import parse_medication_attributes
sample_rx = RXNORM_RECORDS[0]
print({"retrieval": RXNORM_INDEX.retrieve((sample_rx.get("detection_aliases") or [sample_rx["candidate_id"]])[0], top_k=3), "attributes": parse_medication_attributes("aspirin 81 mg po daily")})

## 21. Candidate reranking

**Mục tiêu:** Apply deterministic weighted lexical/character score and output-k policy.  
**Công nghệ:** SequenceMatcher + token overlap + character n-grams; threshold config.  
**Input:** Retriever candidate list.  
**Output:** Reranked candidates.  
**Artifact:** thresholds.json.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
LINKER = EntityLinker(ICD_INDEX, RXNORM_INDEX, CONFIG["candidate_top_k"], CONFIG["candidate_output_k"], CONFIG["thresholds"]["candidate_min_score"])
print({"reranker": "weighted lexical + character", "top_k": CONFIG["candidate_top_k"], "output_k": CONFIG["candidate_output_k"]})

## 22. Relation extraction

**Mục tiêu:** Generate type-compatible same-sentence relation diagnostics.  
**Công nghệ:** RuleRelationExtractor and pair constraints.  
**Input:** Internal entities.  
**Output:** Relation predictions (diagnostics only).  
**Artifact:** diagnostics/relations.json when inference runs.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
RELATION_EXTRACTOR = RuleRelationExtractor(CONFIG["relation_max_distance"])
relations = RELATION_EXTRACTOR.extract(DOCUMENTS[0].raw_text, REFINED_SAMPLE)
print({"sample_relations": len(relations), "submission_key_added": False})

## 23. Integrated pipeline

**Mục tiêu:** Run the complete inference graph using saved artifacts.  
**Công nghệ:** ClinicalNLPPipeline and run_inference.  
**Input:** input.zip and artifacts/.  
**Output:** INTEGRATION_SUMMARY.  
**Artifact:** output/, diagnostics/, output.zip.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
INTEGRATION_SUMMARY = run_inference(PROJECT_ROOT / CONFIG["input_zip"], PROJECT_ROOT / CONFIG["output_dir"], PROJECT_ROOT / CONFIG["artifact_dir"], True, PROJECT_ROOT / CONFIG["diagnostics_dir"], OUTPUT_ARCHIVE_PATH)
print(INTEGRATION_SUMMARY)

## 24. Competition evaluator

**Mục tiêu:** Expose strict and approximate scoring without claiming organizer equivalence.  
**Công nghệ:** Custom strict/greedy matching, WER and Jaccard.  
**Input:** Gold annotations when available.  
**Output:** EVALUATOR_STATUS.  
**Artifact:** No score artifact when gold absent.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
EVALUATOR_STATUS = {"strict": "not_scored", "approximate": "not_scored", "reason": "No ground truth"}
print(EVALUATOR_STATUS)

## 25. Error analysis

**Mục tiêu:** Summarize internal detections and schema drops; never fabricate gold errors.  
**Công nghệ:** Diagnostics aggregation.  
**Input:** diagnostics/*.json.  
**Output:** ERROR_ANALYSIS.  
**Artifact:** diagnostics/run_summary.json.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
summary_path = PROJECT_ROOT / CONFIG["diagnostics_dir"] / "run_summary.json"
ERROR_ANALYSIS = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {"status": "not_run"}
print(ERROR_ANALYSIS)

## 26. Test inference

**Mục tiêu:** Validate every output JSON against every raw input.  
**Công nghệ:** Schema validator and ZIP checks.  
**Input:** output/*.json and input.zip.  
**Output:** TEST_INFERENCE_STATUS.  
**Artifact:** output.zip.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
TEST_INFERENCE_STATUS = {"output_json_count": len(list((PROJECT_ROOT / CONFIG["output_dir"]).glob("*.json"))), "expected": len(DOCUMENTS), "offset_errors": INTEGRATION_SUMMARY["offset_error_count"]}
assert TEST_INFERENCE_STATUS["output_json_count"] == len(DOCUMENTS)
print(TEST_INFERENCE_STATUS)

## 27. Submission generation

**Mục tiêu:** Create and validate output.zip with no nested output directory.  
**Công nghệ:** zipfile and official schema validator.  
**Input:** output/*.json.  
**Output:** output.zip.  
**Artifact:** output.zip.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
import zipfile
with zipfile.ZipFile(OUTPUT_ARCHIVE_PATH) as archive:
    ZIP_NAMES = archive.namelist()
    assert len(ZIP_NAMES) == len(DOCUMENTS)
    assert all(name.startswith("output/") and not name.startswith("output/output/") for name in ZIP_NAMES)
    assert archive.testzip() is None
print({"members": len(ZIP_NAMES), "structure_valid": True, "output_zip": str(OUTPUT_ARCHIVE_PATH)})

## 28. Save/load artifacts

**Mục tiêu:** Reload every inference artifact without training again.  
**Công nghệ:** Artifact-first pipeline construction.  
**Input:** artifacts/ and input.zip.  
**Output:** RELOAD_CHECK.  
**Artifact:** artifacts/model_status.json and KB caches.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
RELOAD_CHECK = reload_equivalence_check(PROJECT_ROOT / CONFIG["input_zip"], PROJECT_ROOT / CONFIG["artifact_dir"])
assert RELOAD_CHECK["equivalent"]
print(RELOAD_CHECK)

## 29. Reproducibility test

**Mục tiêu:** Confirm deterministic seed/config and before/after reload equivalence.  
**Công nghệ:** Seed status, checksums and deterministic rule pipeline.  
**Input:** CONFIG and artifacts.  
**Output:** REPRODUCIBILITY_STATUS.  
**Artifact:** diagnostics/integration_report.json.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
REPRODUCIBILITY_STATUS = {"seed_status": SEED_STATUS, "reload_equivalent": RELOAD_CHECK["equivalent"], "fit_on_private_input": False}
assert REPRODUCIBILITY_STATUS["reload_equivalent"]
print(REPRODUCIBILITY_STATUS)

## 30. Conclusion

**Mục tiêu:** Summarize completed stages, limitations and next organizer confirmation.  
**Công nghệ:** Project state and manifest.  
**Input:** All stage reports.  
**Output:** Final completion summary.  
**Artifact:** README.md and COLAB_RUNBOOK.md.  
**Kiểm tra:** Cell thực thi bên dưới; nếu thiếu annotation, cell ghi rõ trạng thái `not_scored` thay vì bịa metric.


In [ ]:
print({"completed_stages": list(range(1, 10)), "submission_schema_valid": True, "output_zip": str(OUTPUT_ARCHIVE_PATH), "trained_ner": bool(NER_TRAINING_RESULT.get("trained")), "official_mapping": INTEGRATION_SUMMARY.get("official_mapping_status"), "submission_entities": INTEGRATION_SUMMARY.get("submission_entity_count"), "limitation": "Supervised quality metrics require annotated train/validation data."})